# Objetivo
Generar samples y obtener una idea del escalado temporal, todo aprovechando de usar tecnicas de batching programadas; de esta manera generando datos para los modelos de ML clasificativos.

In [1]:
import numpy as np
import pandas as pd
import time
import os
#import psutil
import glob
import pickle
from scipy.stats import qmc
from lib.oracle import OracleExecutor  # assumes your OracleExecutor is in oracle_wrapper.py


In [6]:
# ----- fixed params
epsilon = 0.001
vev = 246

tanbeta_fixed = 1000
sin_betaalpha_fixed = 1



# ---------

def generate_local_variations(
    m_phi_base: float,
    m_A_center: float,
    m12_center: float,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    seed: int = None
    ) -> np.ndarray:
    """
    Generate `batch_size` points where only m_A and m12_2 vary in a small Latin 
    Hypercube around (m_A_center, m12_center), and all other 5 dims are fixed.

    Returns an array of shape (batch_size, 7) with column order:
      [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
    """

    # 1) LatinHypercube in 2D (for m_A and m12)
    sampler = qmc.LatinHypercube(d=2)
    unit = sampler.random(n=batch_size)

    # 2) Scale to [m_A_center±eps_A] × [m12_center±eps_m12]
    bounds = np.array([
        [m_A_center - eps_A, m_A_center + eps_A],
        [m12_center - eps_m12, m12_center + eps_m12]
    ])
    scaled = qmc.scale(unit, bounds[:,0], bounds[:,1])  # shape (batch_size,2)

    # 3) Build the full parameter array, hard-coding the other 5 dims:
    P = np.empty((batch_size, 7), dtype=float)
    P[:, 0] = m_phi_base           # m_phi
    P[:, 1] = scaled[:, 0]         # m_A (varying)
    P[:, 2] = sin_betaalpha_fixed                  # sin(beta - alpha), fixed
    P[:, 3] = tanbeta_fixed              # tan(beta), fixed
    P[:, 4] = 0.1                  # lambda_6, fixed
    P[:, 5] = 0.0                  # lambda_7, fixed
    P[:, 6] = scaled[:, 1]         # m12_2 (varying)

    return P


def get_parameters_from_points(
    csv_path: str,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    seed: int = None
    ) -> np.ndarray:
    """
    Reads `csv_path` containing base points with columns 'Mh2', 'Mh3', 'm12_2' and
    for each row generates `batch_size` local variations via generate_local_variations.
    Returns a combined array of shape (n_rows * batch_size, 7).
    """
    df = pd.read_csv(csv_path)
    all_batches = []
    for idx, row in df.iterrows():
        m_phi_base   = row["Mh2"]    # heavy CP-even Higgs mass as m_phi
        m_A_center   = row["Mh3"]    # CP-odd Higgs mass as m_A
        m12_center   = row["m12_2"]
        # derive unique seed per batch for reproducibility
        batch_seed = None if seed is None else seed + idx
        P = generate_local_variations(
            m_phi_base, m_A_center, m12_center,
            batch_size, eps_A, eps_m12, seed=batch_seed
        )
        all_batches.append(P)
    # stack all batches into one array
    return np.vstack(all_batches)


import numpy as np
from scipy.stats import qmc

def generate_local_variations_phi(
    m_phi_center: float,
    m12_center: float,
    batch_size: int,
    eps_phi: float,
    eps_m12: float,
    m_A_fixed: float = 300.0,
    seed: int = None
) -> np.ndarray:
    """
    Genera `batch_size` puntos variando m_phi y m12_2 en un Latin Hypercube
    alrededor de (m_phi_center, m12_center).
    m_A se fija a `m_A_fixed`. Las otras 4 dimensiones están hardcodeadas.
    Retorna un array de forma (batch_size, 7) con columnas:
    [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
    """
    # 1) Muestreo LH en 2D para (m_phi, m12_2)
    sampler = qmc.LatinHypercube(d=2)
    unit = sampler.random(n=batch_size)

    # 2) Escalado a [m_phi_center±eps_phi] × [m12_center±eps_m12]
    bounds = np.array([
        [m_phi_center - eps_phi, m_phi_center + eps_phi],
        [m12_center  - eps_m12,  m12_center  + eps_m12]
    ])
    scaled = qmc.scale(unit, bounds[:,0], bounds[:,1])  # shape (batch_size,2)

    # 3) Construir el array de parámetros
    P = np.empty((batch_size, 7), dtype=float)
    P[:, 0] = scaled[:, 0]       # m_phi (variado)
    P[:, 1] = m_A_fixed           # m_A (fijo)
    P[:, 2] = sin_betaalpha_fixed                 # sin(beta - alpha)
    P[:, 3] = tanbeta_fixed             # tan(beta)
    P[:, 4] = 0.1                 # lambda6
    P[:, 5] = 0.0                 # lambda7
    P[:, 6] = scaled[:, 1]       # m12_2 (variado)

    return P

def get_parameters_from_points_phi(
    csv_path: str,
    batch_size: int,
    eps_phi: float,
    eps_m12: float,
    m_A_fixed: float = 300.0,
    seed: int = None
) -> np.ndarray:
    """
    Lee `csv_path` con columnas 'Mh2' (m_phi_center) y 'm12_2'.
    Para cada fila genera un batch con `generate_local_variations_phi`.
    """
    import pandas as pd
    df = pd.read_csv(csv_path)
    all_batches = []
    for idx, row in df.iterrows():
        m_phi_center = row["Mh2"]
        m12_center   = row["m12_2"]
        batch_seed   = None if seed is None else seed + idx
        P = generate_local_variations_phi(
            m_phi_center,
            m12_center,
            batch_size,
            eps_phi,
            eps_m12,
            m_A_fixed=m_A_fixed,
            seed=batch_seed
        )
        # DEBUG: validación rápida
        # assert np.all(P[:,1] == m_A_fixed), "m_A no está fijo a 300"
        all_batches.append(P)

    return np.vstack(all_batches)

# Ejemplo de uso:
# >>> params = get_parameters_from_points_phi(
#       "puntos_base.csv",
#       batch_size=100,
#       eps_phi=0.5,
#       eps_m12=1.0,
#       m_A_fixed=300.0,
#       seed=42
#     )
# >>> print(params.shape)  # debería ser (n_rows * 100, 7)



# Prepare executor
executor = OracleExecutor(nthreads=4)



In [7]:
import os
import glob
import time
import pickle
import pandas as pd
from typing import Literal
from lib.oracle import OracleExecutor

def multiple_runs(
    csv_path: str,
    N_repeat_runs: int,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    outdir: str,
    executor: OracleExecutor,
    variation_mode: Literal['mA', 'mPhi'] = 'mA',
    eps_phi: float = None,
    base_seed: int = 42,
    output_style: str = 'tan1e4sinba100'
):
    """
    Ejecuta varias corridas locales LH. Dependiendo de `variation_mode`:
      - 'mA': varía (m_A, m12_2), fija m_phi (usa eps_A)
      - 'mPhi': varía (m_phi, m12_2), fija m_A=300, usa eps_phi
    """
    # Validaciones básicas
    if variation_mode == 'mPhi' and eps_phi is None:
        raise ValueError("Para mode='mPhi' debes proporcionar eps_phi")
    
    df = pd.read_csv(csv_path)
    n_runs = len(df)

    # Estimación de tiempo
    time_per_point = 415.7 / 15_000
    total_points = N_repeat_runs * n_runs * batch_size
    pred_mins = total_points * time_per_point / 60
    print(f"Estimado: {pred_mins:.1f} min (~{pred_mins/60:.2f} h) para {total_points} puntos")

    os.makedirs(outdir, exist_ok=True)

    for epoch in range(N_repeat_runs):
        for j, row in df.iterrows():
            m_phi_base = float(row["Mh2"])
            m_A_center = float(row["Mh3"])
            m12_center = float(row["m12_2"])
            # Semilla reproducible única
            seed = base_seed + epoch * n_runs + j

            # Seleccionar la función de variación según el modo
            if variation_mode == 'mA':
                param_list = generate_local_variations(
                    m_phi_base=m_phi_base,
                    m_A_center=m_A_center,
                    m12_center=m12_center,
                    batch_size=batch_size,
                    eps_A=eps_A,
                    eps_m12=eps_m12,
                    seed=seed
                )
            else:  # 'mPhi'
                param_list = generate_local_variations_phi(
                    m_phi_center=m_phi_base,
                    m12_center=m12_center,
                    batch_size=batch_size,
                    eps_phi=eps_phi,
                    eps_m12=eps_m12,
                    m_A_fixed=m_A_center,  # o un valor fijo, p.ej. 300
                    seed=seed
                )

            # Índice de batch basado en archivos existentes
            existing = sorted(glob.glob(os.path.join(outdir, "batch_*.pkl")))
            batch_idx = len(existing) + 1

            # Ejecución
            t0 = time.perf_counter()
            results = executor.map(param_list.tolist(), use_threads=True)
            dt = time.perf_counter() - t0

            # Guardar resultados
            fname = f"batch_{batch_idx}_{int(m_phi_base)}_{output_style}.pkl"
            fout = os.path.join(outdir, fname)
            with open(fout, "wb") as f:
                pickle.dump({"params": param_list, "results": results}, f)

            print(f"[Run {j+1}/{n_runs}] batch {batch_idx} "
                  f"mode={variation_mode} saved in {dt:.1f}s → {fout}")

        print(f"[Epoch {epoch+1}/{N_repeat_runs}] completado")
    print("Todas las ejecuciones finalizadas.")


# Runs

In [8]:
import os
import glob
import time
import pickle

import numpy as np
from scipy.stats import qmc
import pandas as pd

# Asegúrate de importar tu executor y multiple_runs
from lib.oracle import OracleExecutor

# ── Parámetros de usuario ─────────────────────────────────────────────────────
CSV_PATH      = "valid_points_lhe/valid_points.csv"
OUTDIR        = "data_batches"
N_RUNS        = 2           # cuántos puntos base correr
BATCH_SIZE    = 2_000       # puntos por run
EPS_A         = 0         # ± variación en m_A (no usado en modo mPhi)
EPS_PHI       = 0.1         # ± variación en m_phi (nuevo)
EPS_M12       = 0.0002         # ± variación en m12^2
#SEED          = 100         # semilla base

# ── Preparación ────────────────────────────────────────────────────────────────
os.makedirs(OUTDIR, exist_ok=True)
existing_batches = sorted(glob.glob(f"{OUTDIR}/batch_*.pkl"))
print(f"Próximo batch id: {len(existing_batches) + 1}")

executor = OracleExecutor(nthreads=4)

# ── Ejecución ─────────────────────────────────────────────────────────────────
multiple_runs(
    csv_path=CSV_PATH,
    N_repeat_runs=N_RUNS,
    batch_size=BATCH_SIZE,
    eps_A=EPS_A,                   # ignorado en 'mPhi'
    eps_m12=EPS_M12,
    outdir=OUTDIR,
    executor=executor,
    variation_mode='mPhi',         # ¡aquí cambiamos el modo!
    eps_phi=EPS_PHI,               # ± rango para m_phi
    output_style='tan1e3sinba1'
)



print("Run variando m_phi completada.")


Próximo batch id: 37
Estimado: 33.3 min (~0.55 h) para 72000 puntos
[Run 1/18] batch 37 mode=mPhi saved in 34.3s → data_batches/batch_37_130_tan1e3sinba1.pkl
[Run 2/18] batch 38 mode=mPhi saved in 33.9s → data_batches/batch_38_139_tan1e3sinba1.pkl
[Run 3/18] batch 39 mode=mPhi saved in 34.3s → data_batches/batch_39_150_tan1e3sinba1.pkl
[Run 4/18] batch 40 mode=mPhi saved in 34.4s → data_batches/batch_40_160_tan1e3sinba1.pkl
[Run 5/18] batch 41 mode=mPhi saved in 34.6s → data_batches/batch_41_170_tan1e3sinba1.pkl
[Run 6/18] batch 42 mode=mPhi saved in 35.6s → data_batches/batch_42_180_tan1e3sinba1.pkl
[Run 7/18] batch 43 mode=mPhi saved in 36.4s → data_batches/batch_43_190_tan1e3sinba1.pkl
[Run 8/18] batch 44 mode=mPhi saved in 36.0s → data_batches/batch_44_200_tan1e3sinba1.pkl
[Run 9/18] batch 45 mode=mPhi saved in 36.7s → data_batches/batch_45_210_tan1e3sinba1.pkl
[Run 10/18] batch 46 mode=mPhi saved in 36.3s → data_batches/batch_46_220_tan1e3sinba1.pkl
[Run 11/18] batch 47 mode=mPhi 

# Merging

In [ ]:
# ------------------------
# Merge old batches if they exceed size threshold
# ------------------------
def merge_batches(folder, batch_prefix="batch_", merged_prefix="merged_", max_size_mb=30):
    # Count existing merged files to avoid overwrite
    existing_merged = sorted(glob.glob(f"{folder}/{merged_prefix}*.pkl"))
    merge_idx = len(existing_merged) + 1
    
    # Only consider raw batch files
    batch_files = sorted(glob.glob(f"{folder}/{batch_prefix}*.pkl"))
    acc_size = 0
    group = []

    for fp in batch_files:
        fsize = os.path.getsize(fp)
        if (acc_size + fsize) / (1024**2) > max_size_mb and group:
            # Merge current group
            merged_data = []
            for gfp in group:
                with open(gfp, "rb") as gf:
                    merged_data.append(pickle.load(gf))
                os.remove(gfp)
            mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
            with open(mout, "wb") as mf:
                pickle.dump(merged_data, mf)
            print(f"Merged {len(group)} batches into {mout}")
            merge_idx += 1
            group, acc_size = [], 0

        group.append(fp)
        acc_size += fsize

    # Merge any remaining files
    if group:
        merged_data = []
        for gfp in group:
            with open(gfp, "rb") as gf:
                merged_data.append(pickle.load(gf))
            os.remove(gfp)
        mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
        with open(mout, "wb") as mf:
            pickle.dump(merged_data, mf)
        print(f"Merged {len(group)} batches into {mout}")
    return merged_data

# Call merge

OUTDIR = "data_batches"
merged_data = merge_batches(OUTDIR, merged_prefix="merged_tan1e3_")
# merged_data

# Testing Speed

In [ ]:
# Define the sampling sizes
sample_sizes = [1, 10, 100, 1_000, 10_000]

for n in sample_sizes:
    # Generate Latin Hypercube samples in [0,1]^7, then scale
    sampler = qmc.LatinHypercube(d=7)
    sample_unit = sampler.random(n)
    param_list = qmc.scale(sample_unit, param_bounds[:,0], param_bounds[:,1])
    
    # Measure memory before run
    process = psutil.Process()
    mem_before = process.memory_info().rss
    
    # Run and time
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=False)
    t1 = time.perf_counter()
    
    mem_after = process.memory_info().rss
    delta_mem = (mem_after - mem_before) / (1024**2)  # in MB
    
    # Save raw results for this batch
    with open(f"oracle_results_{n}.pkl", "wb") as f:
        pickle.dump(results, f)
    
    # Record performance
    perf_records.append({
        "n_points": n,
        "time_sec": t1 - t0,
        "mem_delta_MB": delta_mem
    })
    print(f"Completed batch {n}: time={t1-t0:.2f}s, memory Δ={delta_mem:.1f}MB")

# Save performance table
df_perf = pd.DataFrame(perf_records)
df_perf.to_csv("performance_scaling.csv", index=False)

